# Resultaten — dataset, model en mutaties in één overzicht

Draai de cellen van boven naar beneden (VS Code: kies rechtsboven de **.venv**-kernel).
Elke sectie meldt het netjes als het bijbehorende onderdeel nog niet gedraaid is —
je kunt dit notebook dus op elk moment in het project openen.

In [ ]:
import json, random, sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw
from IPython.display import display

REPO = Path.cwd().resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'scripts'))
from common import KLEUREN, laad_config

cfg = laad_config(REPO / 'config.yaml')
KLASSEN = cfg['dataset']['klassen']
DATA = REPO / cfg['paden']['data']
TILES = DATA / 'tiles'
print('Repo:', REPO)
print('Klassen:', ', '.join(KLASSEN))

## Labelstand per klasse

Zwakke labels (BAG) en handmatige annotaties samen — dit gaat de training in.
Richtlijn: ~300 boxen per klasse voor een bruikbare eerste versie.

In [ ]:
telling = Counter()
labels_map = TILES / 'labels'
if labels_map.exists():
    for p in labels_map.glob('*.txt'):
        for regel in p.read_text(encoding='utf-8').splitlines():
            telling[int(regel.split()[0])] += 1
geannoteerd_pad = TILES / 'geannoteerd.json'
geannoteerd = json.loads(geannoteerd_pad.read_text()) if geannoteerd_pad.exists() else []

if not telling:
    print('Nog geen labels — draai eerst .\\stappen\\1-voorbereiden.ps1')
else:
    aantallen = [telling.get(i, 0) for i in range(len(KLASSEN))]
    fig, ax = plt.subplots(figsize=(8, 3.6))
    posities = range(len(KLASSEN))
    ax.barh(posities, aantallen, height=0.55,
            color=[KLEUREN[i % len(KLEUREN)] for i in posities])
    ax.set_yticks(posities, KLASSEN)
    ax.invert_yaxis()
    for i, n in enumerate(aantallen):
        ax.text(n + max(aantallen) * 0.015, i, str(n), va='center',
                color='#555555', fontsize=9)
    ax.set_title(f'Labels per klasse — {len(geannoteerd)} tegels handmatig geannoteerd',
                 loc='left', fontsize=11)
    ax.set_xlabel('aantal boxen')
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(length=0)
    plt.tight_layout()
    plt.show()

## Tegels met labels

Vier willekeurige tegels met de huidige boxen erop getekend, in de klassenkleuren.

In [ ]:
def teken_tegel(tegel_id, index):
    beeld = Image.open(TILES / index[tegel_id]['image']).convert('RGB')
    teken = ImageDraw.Draw(beeld)
    px = index[tegel_id]['px']
    label_pad = TILES / 'labels' / f'{tegel_id}.txt'
    if label_pad.exists():
        for regel in label_pad.read_text(encoding='utf-8').splitlines():
            d = regel.split()
            idx = int(d[0])
            cx, cy, b, h = (float(v) for v in d[1:])
            teken.rectangle([(cx - b / 2) * px, (cy - h / 2) * px,
                             (cx + b / 2) * px, (cy + h / 2) * px],
                            outline=KLEUREN[idx % len(KLEUREN)], width=3)
    return beeld

index_pad = TILES / 'tiles.json'
if not index_pad.exists():
    print('Nog geen tegels — draai eerst .\\stappen\\1-voorbereiden.ps1')
else:
    index = json.loads(index_pad.read_text(encoding='utf-8'))
    keuze = random.Random(0).sample(sorted(index), k=min(4, len(index)))
    fig, assen = plt.subplots(2, 2, figsize=(10, 10.6))
    for ax, tegel_id in zip(assen.flat, keuze):
        ax.imshow(teken_tegel(tegel_id, index))
        ax.set_title(tegel_id, fontsize=9)
        ax.axis('off')
    handvatten = [mpatches.Patch(color=KLEUREN[i % len(KLEUREN)], label=k)
                  for i, k in enumerate(KLASSEN)]
    fig.legend(handles=handvatten, loc='lower center', ncol=5, frameon=False, fontsize=9)
    plt.tight_layout(rect=(0, 0.05, 1, 1))
    plt.show()

## Modelvoorspellingen

Het nieuwste model uit `data/models/` losgelaten op willekeurige tegels.

In [ ]:
modellen_map = DATA / 'models'
modellen = (sorted(modellen_map.glob('*.pt'), key=lambda p: p.stat().st_mtime)
            if modellen_map.exists() else [])
if not modellen:
    print('Nog geen model in data/models — draai .\\stappen\\4-trainen.ps1 (en 5-model-ophalen).')
elif not index_pad.exists():
    print('Nog geen tegels om op te voorspellen.')
else:
    try:
        from ultralytics import YOLO
        model = YOLO(str(modellen[-1]))
        keuze = random.Random(1).sample(sorted(index), k=min(4, len(index)))
        fig, assen = plt.subplots(2, 2, figsize=(10, 10.4))
        for ax, tegel_id in zip(assen.flat, keuze):
            res = model.predict(str(TILES / index[tegel_id]['image']),
                                conf=0.25, verbose=False)[0]
            ax.imshow(res.plot()[..., ::-1])
            ax.set_title(f'{tegel_id} — {len(res.boxes)} detecties', fontsize=9)
            ax.axis('off')
        fig.suptitle(f'Voorspellingen van {modellen[-1].name}', fontsize=11)
        plt.tight_layout()
        plt.show()
    except ImportError:
        print('ultralytics ontbreekt — pip install -r requirements-train.txt')

## Trainingscurves en confusion matrix (Kaggle)

Uit de laatst gedownloade trainingsoutput (`data/kaggle_output/`).

In [ ]:
kaggle_out = DATA / 'kaggle_output'
plaatjes = (sorted(kaggle_out.rglob('results.png'))
            + sorted(kaggle_out.rglob('confusion_matrix_normalized.png'))
            if kaggle_out.exists() else [])
if not plaatjes:
    print('Geen Kaggle-output gevonden — .\\stappen\\5-model-ophalen.ps1 downloadt ook de curves.')
else:
    for p in plaatjes:
        print(p.relative_to(kaggle_out))
        beeld = Image.open(p)
        beeld.thumbnail((1200, 1200))
        display(beeld)

## Verschilcomposieten (mutatiedetectie)

Drieluiken oud | nieuw | composiet — rood = verdwenen, cyaan = nieuw, grijs = ongewijzigd.
Zie [docs/mutatiedetectie.md](../docs/mutatiedetectie.md).

In [ ]:
verander = DATA / 'verander'
drieluiken = sorted(verander.rglob('preview/*.jpg')) if verander.exists() else []
if not drieluiken:
    print('Nog geen composieten — zie docs/mutatiedetectie.md '
          '(stap 02 --laag <jaargang> en daarna stap 11).')
else:
    for p in drieluiken[:4]:
        print(f'{p.parent.parent.name} — {p.stem}')
        beeld = Image.open(p)
        beeld.thumbnail((1400, 1400))
        display(beeld)

---
*Bevat gegevens van PDOK: Luchtfoto Beeldmateriaal Nederland (CC-BY 4.0) en de BAG.*